In [ ]:
# ── 1. 环境导入 ──────────────────────────────────────────────────────────────
import sys, os, math, time, re
sys.path.insert(0, os.path.abspath('.'))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

print(f'PyTorch {torch.__version__}')
device = (
    'cuda' if torch.cuda.is_available()
    else 'mps'  if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'计算设备: {device}')

In [ ]:
# ── 2. 加载 WMT14 英德翻译数据集 ─────────────────────────────────────────────
# split 切片语法直接告知 HuggingFace 只下载对应分片，不会拉取全量数据
import os
os.makedirs('data', exist_ok=True)
os.environ['HF_DATASETS_CACHE'] = os.path.abspath('data')

from datasets import load_dataset

print('下载 WMT14 de-en（仅训练集前 10%，验证/测试集保持完整）...')
train_raw = load_dataset('wmt14', 'de-en', split='train[:20%]',  trust_remote_code=True, cache_dir='data')
valid_raw = load_dataset('wmt14', 'de-en', split='validation',   trust_remote_code=True, cache_dir='data')
test_raw  = load_dataset('wmt14', 'de-en', split='test[:3000]',  trust_remote_code=True, cache_dir='data')

print(f'Train: {len(train_raw):,}  Valid: {len(valid_raw):,}  Test: {len(test_raw):,}')
print('样本示例:', train_raw[0]['translation'])


In [ ]:
# ── 3. BPE 分词器（与原论文一致）────────────────────────────────────────────
# Vaswani et al. (2017) 使用共享源/目标 BPE 词表，约 37000 个 subword 单元
# 这里用 HuggingFace tokenizers 库复现相同流程：
#   1. 在训练集上训练 BPE（英德共享词表）
#   2. 词表大小 37000，与原论文一致
#   3. 模型文件持久化到 data/ 目录，避免重复训练

import os, json
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing

PAD_TOKEN = '<pad>'
BOS_TOKEN = '<bos>'
EOS_TOKEN = '<eos>'
UNK_TOKEN = '<unk>'
SPECIAL   = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

BPE_VOCAB_SIZE = 37_000   # 原论文词表大小
BPE_MODEL_PATH = 'data/bpe_wmt14.json'

def get_corpus_iterator(dataset):
    for ex in dataset:
        t = ex['translation']
        yield t['en']
        yield t['de']

if os.path.exists(BPE_MODEL_PATH):
    print(f'从 {BPE_MODEL_PATH} 加载已有 BPE 模型...')
    tokenizer = Tokenizer.from_file(BPE_MODEL_PATH)
else:
    print('在训练集上训练 BPE（英德共享词表）...')
    tokenizer = Tokenizer(BPE(unk_token=UNK_TOKEN))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(
        vocab_size=BPE_VOCAB_SIZE,
        special_tokens=SPECIAL,
        min_frequency=2,
        show_progress=True,
    )
    tokenizer.train_from_iterator(get_corpus_iterator(train_raw), trainer=trainer)
    tokenizer.save(BPE_MODEL_PATH)
    print(f'BPE 模型已保存至 {BPE_MODEL_PATH}')

# 注入 BOS/EOS 后处理（训练后 post_processor 无法直接 save/load，手动处理）
VOCAB      = tokenizer.get_vocab()
PAD_IDX    = VOCAB[PAD_TOKEN]   # 0
BOS_IDX    = VOCAB[BOS_TOKEN]   # 1
EOS_IDX    = VOCAB[EOS_TOKEN]   # 2
SRC_VOCAB  = tokenizer.get_vocab_size()
TGT_VOCAB  = SRC_VOCAB          # 英德共享词表

print(f'BPE 词表大小: {SRC_VOCAB}')
print(f'PAD={PAD_IDX}  BOS={BOS_IDX}  EOS={EOS_IDX}')


In [ ]:
# ── 4. 编码函数与 Dataset ─────────────────────────────────────────────────────
MAX_SRC_LEN = 128   # BPE subword 序列最大长度（原论文约 100 token）
MAX_TGT_LEN = 128

def encode(text, max_len):
    ids = tokenizer.encode(text).ids
    return ids[:max_len]

class TranslationDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        pair    = self.examples[idx]['translation']
        src_ids = [BOS_IDX] + encode(pair['en'], MAX_SRC_LEN - 2) + [EOS_IDX]
        tgt_ids = [BOS_IDX] + encode(pair['de'], MAX_TGT_LEN - 2) + [EOS_IDX]
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)

def collate_fn(batch):
    src_list, tgt_list = zip(*batch)
    src = nn.utils.rnn.pad_sequence(src_list, batch_first=True, padding_value=PAD_IDX)
    tgt = nn.utils.rnn.pad_sequence(tgt_list, batch_first=True, padding_value=PAD_IDX)
    return src, tgt

BATCH_SIZE   = 64
train_loader = DataLoader(TranslationDataset(train_raw), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=0)
valid_loader = DataLoader(TranslationDataset(valid_raw), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(TranslationDataset(test_raw),  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f'Train batches: {len(train_loader)}  Valid: {len(valid_loader)}  Test: {len(test_loader)}')
src_b, tgt_b = next(iter(train_loader))
print(f'src batch shape: {src_b.shape}  tgt batch shape: {tgt_b.shape}')


In [ ]:
# ── 5. 构建 Encoder-Decoder Transformer 模型 ─────────────────────────────────
# 超参数参考原论文 Base 规模；资源有限时使用小规模（d_model=256, layers=3）
from transformer import Transformer

D_MODEL        = 256
NUM_HEADS      = 8
NUM_ENC_LAYERS = 3
NUM_DEC_LAYERS = 3
D_FF           = 512
DROPOUT        = 0.1
MAX_LEN        = max(MAX_SRC_LEN, MAX_TGT_LEN) + 10

model = Transformer(
    src_vocab_size=SRC_VOCAB,
    tgt_vocab_size=TGT_VOCAB,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_encoder_layers=NUM_ENC_LAYERS,
    num_decoder_layers=NUM_DEC_LAYERS,
    d_ff=D_FF,
    max_len=MAX_LEN,
    dropout=DROPOUT,
    pad_idx=PAD_IDX,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'模型参数量: {n_params:,}')

with torch.no_grad():
    _src = src_b[:2].to(device)
    _tgt = tgt_b[:2, :-1].to(device)
    _out = model(_src, _tgt)
    print(f'前向传播输出形状: {_out.shape}')  # (2, tgt_len-1, TGT_VOCAB)


In [ ]:
# ── 6. 损失函数、优化器与学习率调度 ──────────────────────────────────────────
# 原论文使用 Noam 学习率调度（warmup + 逆平方根衰减）
# 这里使用 CosineAnnealingLR 简化实现，效果相近

EPOCHS    = 10
LR        = 1e-3
CLIP_GRAD = 1.0
LOG_EVERY = 100
WARMUP    = 400   # warmup 步数

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9)

# Noam 调度：lr = d_model^{-0.5} * min(step^{-0.5}, step * warmup^{-1.5})
def noam_lambda(step):
    step = max(step, 1)
    return D_MODEL ** -0.5 * min(step ** -0.5, step * WARMUP ** -1.5)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=noam_lambda)
print('损失函数、优化器与 Noam 调度器已初始化')

In [ ]:
# ── 7. 训练与评估函数 ─────────────────────────────────────────────────────────
# train_epoch: teacher-forcing 训练，解码器输入为 tgt[:,:-1]，标签为 tgt[:,1:]
# evaluate:    无梯度推理，返回平均损失

def train_epoch(model, loader, optimizer, criterion, scheduler, clip, device, log_every):
    model.train()
    total_loss = total_tok = 0
    t0 = time.time()
    for step, (src, tgt) in enumerate(loader, 1):
        src, tgt = src.to(device), tgt.to(device)
        tgt_in  = tgt[:, :-1]   # decoder 输入：去掉最后一个 token
        tgt_out = tgt[:, 1:]    # 标签：去掉第一个 token（BOS）

        logits = model(src, tgt_in)                          # (B, T-1, TGT_VOCAB)
        loss   = criterion(logits.reshape(-1, TGT_VOCAB), tgt_out.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        scheduler.step()

        n = (tgt_out != PAD_IDX).sum().item()
        total_loss += loss.item() * n
        total_tok  += n

        if step % log_every == 0:
            avg = total_loss / total_tok
            lr  = scheduler.get_last_lr()[0]
            print(f'  step {step:5d}/{len(loader)}  loss={avg:.4f}  ppl={math.exp(min(avg,10)):.1f}  lr={lr:.2e}  {time.time()-t0:.0f}s')
    return total_loss / total_tok


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = total_tok = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        tgt_in  = tgt[:, :-1]
        tgt_out = tgt[:, 1:]
        logits = model(src, tgt_in)
        loss   = criterion(logits.reshape(-1, TGT_VOCAB), tgt_out.reshape(-1))
        n = (tgt_out != PAD_IDX).sum().item()
        total_loss += loss.item() * n
        total_tok  += n
    avg = total_loss / total_tok
    return avg, math.exp(min(avg, 10))

print('训练/评估函数已定义')

In [ ]:
# ── 8. 训练循环 ───────────────────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'val_ppl': []}
best_val_loss = float('inf')
CKPT_PATH = 'transformer_mt_best.pt'

for epoch in range(1, EPOCHS + 1):
    print(f'\n{"="*60}\nEpoch {epoch}/{EPOCHS}')
    t0 = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, criterion, scheduler, CLIP_GRAD, device, LOG_EVERY)
    val_loss, val_ppl = evaluate(model, valid_loader, criterion, device)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_ppl'].append(val_ppl)
    print(f'>>> train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_ppl={val_ppl:.2f} | {time.time()-t0:.0f}s')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CKPT_PATH)
        print(f'    -> 保存最优模型 (val_loss={val_loss:.4f})')

print(f'\n训练完成。最优验证集 Loss: {best_val_loss:.4f}')

In [ ]:
# ── 9. 训练曲线可视化 ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

ep = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ep, history['train_loss'], 'o-', label='Train')
axes[0].plot(ep, history['val_loss'],   's-', label='Val')
axes[0].set_title('Loss 曲线'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(ep, history['val_ppl'], '^-', color='orange')
axes[1].set_title('验证集 Perplexity'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('PPL')

plt.tight_layout()
plt.savefig('training_curve.png', dpi=120)
plt.show()
print('图像已保存至 training_curve.png')

In [ ]:
# ── 10. 测试集最终评估（Loss / PPL）──────────────────────────────────────────
model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
test_loss, test_ppl = evaluate(model, test_loader, criterion, device)

print('\n===== 测试集结果 =====')
print(f'  Loss       : {test_loss:.4f}')
print(f'  Perplexity : {test_ppl:.2f}')

In [ ]:
# ── 11. 贪心解码翻译示例 ──────────────────────────────────────────────────────
@torch.no_grad()
def greedy_translate(model, src_sentence, max_len=128):
    model.eval()
    src_ids = [BOS_IDX] + encode(src_sentence, MAX_SRC_LEN - 2) + [EOS_IDX]
    src = torch.tensor([src_ids], dtype=torch.long, device=device)

    src_mask   = model.make_src_mask(src)
    enc_output = model.encode(src, src_mask)

    tgt = torch.tensor([[BOS_IDX]], dtype=torch.long, device=device)
    for _ in range(max_len):
        tgt_mask = model.make_tgt_mask(tgt)
        dec_out  = model.decode(tgt, enc_output, src_mask, tgt_mask)
        logits   = model.fc_out(dec_out[:, -1, :])
        next_id  = logits.argmax(dim=-1, keepdim=True)
        if next_id.item() == EOS_IDX:
            break
        tgt = torch.cat([tgt, next_id], dim=1)

    token_ids = tgt[0, 1:].tolist()   # 去掉 BOS
    return tokenizer.decode(token_ids)


samples = [
    'The cat sat on the mat .',
    'A man is walking in the park .',
    'The government announced new economic policies .',
]
for s in samples:
    print(f'English : {s}')
    print(f'German  : {greedy_translate(model, s)}')
    print('-' * 60)
